In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. NNLM (Neural Network Language Model) 클래스 정의
# ==========================================
class NNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size, hidden_dim):
        """
        NNLM 신경망의 레이어들을 초기화합니다.
        
        Args:
            vocab_size (int): 단어 집합(Vocabulary)의 전체 단어 개수
            embedding_dim (int): 원-핫 벡터를 투사할 밀집 임베딩 벡터의 차원 수
            block_size (int): 예측에 사용할 이전 단어의 개수 (Context Window Size = N - 1)
            hidden_dim (int): 은닉층(Hidden Layer)의 노드(뉴런) 개수
        """
        # 부모 클래스인 nn.Module의 초기화 메서드를 호출합니다.
        super(NNLM, self).__init__()
        
        # 1. 단어 인덱스를 embedding_dim 차원의 밀집 벡터로 변환하는 룩업 테이블(Look-up Table)입니다.
        # 단어장의 각 단어마다 고유한 임베딩 벡터가 매핑되며, 학습 과정에서 업데이트됩니다.
        self.C = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. 은닉층(Hidden Layer) 선형 결합 레이어입니다.
        # 이전 block_size개의 단어 임베딩이 하나로 이어붙여지므로(Concatenate),
        # 입력 차원은 (block_size * embedding_dim)이 됩니다.
        self.hidden = nn.Linear(block_size * embedding_dim, hidden_dim)
        
        # Bengio et al. (2003) 논문에서 사용한 비선형 활성화 함수 Tanh입니다.
        # 은닉층 연산 결과에 비선형성을 부여합니다.
        self.tanh = nn.Tanh()
        
        # 3. 은닉층의 출력을 받아 전체 단어장(vocab_size) 차원의 로짓(Logit)으로 변환하는 출력 레이어입니다.
        # 이 결과값에 Softmax를 취하면 다음 단어가 될 각 단어별 확률이 계산됩니다.
        self.output = nn.Linear(hidden_dim, vocab_size)
        
        # 4. Bengio 논문의 원본 구조에 존재하는 Direct Connection(직결 연결) 레이어입니다.
        # 은닉층을 거치지 않고, 임베딩 결합 벡터에서 바로 출력층으로 연결되는 가중치(W_x) 역할을 합니다.
        # 편향(bias)을 중복으로 더하지 않기 위해 bias=False로 설정합니다.
        self.direct = nn.Linear(block_size * embedding_dim, vocab_size, bias=False)

    def forward(self, x):
        """
        순전파(Forward Propagation) 연산을 수행합니다.
        
        Args:
            x (Tensor): [batch_size, block_size] 형태의 이전 단어 인덱스 텐서
            
        Returns:
            logits (Tensor): [batch_size, vocab_size] 형태의 다음 단어 예측 로짓
        """
        # 입력된 단어 인덱스들을 룩업하여 임베딩 벡터로 변환합니다.
        # 변환 후 텐서 크기: [batch_size, block_size, embedding_dim]
        embeds = self.C(x)
        
        # (N-1)개의 이전 단어 임베딩 벡터들을 차원 방향으로 평탄화하여 하나로 이어붙입니다(Concatenate).
        # 변환 후 텐서 크기: [batch_size, block_size * embedding_dim]
        concat_embeds = embeds.view(embeds.size(0), -1)
        print(f'embeds: {embeds}') # Embedding Vectors (17, 3, 8)
        print(f'concat_embeds: {concat_embeds}') # Concatenated Embeddings (17, 8)

        # 선형 결합(self.hidden) 후 Tanh 활성화 함수를 통과시켜 은닉 상태(h)를 계산합니다.
        # 변환 후 텐서 크기: [batch_size, hidden_dim]
        h = self.tanh(self.hidden(concat_embeds))
        
        # Bengio 논문의 수식 y = b + Wx + U*tanh(d + Hx)를 계산합니다.
        # self.output(h)는 은닉층을 거친 경로(U*tanh...)를, self.direct(concat_embeds)는 직결 경로(Wx)를 의미합니다.
        # 변환 후 텐서 크기: [batch_size, vocab_size]
        logits = self.output(h) + self.direct(concat_embeds)
        
        # Softmax 직전의 값인 로짓(Logits)을 반환합니다.
        return logits


# ==========================================
# 2. 텍스트 데이터를 슬라이딩 윈도우로 전처리하는 함수
# ==========================================
def make_dataset(text, word2idx, block_size):
    """
    원문 텍스트를 입력 받아 NNLM 학습용 (입력 문맥 X, 정답 단어 Y) 텐서 쌍으로 변환합니다.
    """
    # 텍스트를 공백 기준으로 나누어 단어 리스트로 만듭니다.
    words = text.split()
    print(f'words: {words}')
    print(f'len(words): {len(words)}')
    
    # 입력 문맥(X)과 타겟 단어(Y)를 담을 리스트 초기화
    X, Y = [], []
    
    # 전체 단어 시퀀스를 순회하며 슬라이딩 윈도우(Sliding Window)를 적용합니다.
    for i in range(len(words) - block_size):
        # 현재 위치 i부터 block_size(N-1)개까지의 이전 단어들을 정수 인덱스 리스트로 추출합니다.
        context = [word2idx[w] for w in words[i : i + block_size]]
        
        # 문맥 단어들 바로 다음에 오는 예측 대상 정답 단어의 인덱스를 추출합니다.
        target = word2idx[words[i + block_size]]
        
        # 리스트에 각각 추가합니다.
        X.append(context)
        Y.append(target)
        
    # PyTorch 모델에 입력할 수 있도록 정수형(Long) 텐서로 변환하여 반환합니다.
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)


# ==========================================
# 3. 모델 학습 및 검증 메인 파이프라인
# ==========================================
if __name__ == "__main__":
    # 무작위 추출 시 일관된 결과를 얻기 위해 PyTorch 난수 생성 시드를 고정합니다.
    torch.manual_seed(42)

    # NNLM 학습에 사용할 예시 코퍼스(Corpus) 문장입니다.
    sample_text = (
        "the neural network language model predicts the next word "
        "given a sequence of previous words using embedding vectors"
    )

    # 예시 문장을 단어 단위로 쪼개고 중복을 제거한 뒤 정렬하여 단어장 토큰 목록을 만듭니다.
    tokens = sorted(list(set(sample_text.split())))
    
    # 단어장의 전체 고유 단어 개수를 구합니다.
    vocab_size = len(tokens)
    
    # 단어를 정수 인덱스로 매핑하는 사전(Dictionary)을 생성합니다.
    word2idx = {w: i for i, w in enumerate(tokens)}
    
    # 정수 인덱스를 다시 단어로 복원할 때 사용하는 역방향 사전을 생성합니다.
    idx2word = {i: w for i, w in enumerate(tokens)}

    # 이전 몇 개의 단어(Context)를 보고 다음 단어를 예측할지 설정합니다. (N-1 = 3 -> N = 4)
    BLOCK_SIZE = 3
    
    # 각 단어를 몇 차원의 밀집 벡터로 임베딩할지 설정합니다.
    EMBEDDING_DIM = 8
    
    # 은닉층(Hidden Layer) 내부의 노드 개수를 설정합니다.
    HIDDEN_DIM = 16
    
    # 가중치 업데이트에 사용할 학습률(Learning Rate)을 설정합니다.
    LEARNING_RATE = 0.01
    
    # 전체 데이터셋을 총 몇 번 반복 학습할지 결정합니다.
    EPOCHS = 200

    # 텍스트 데이터로부터 NNLM 학습용 데이터셋(X: 문맥 인덱스, Y: 정답 인덱스)을 생성합니다.
    X, Y = make_dataset(sample_text, word2idx, BLOCK_SIZE)

    # 하이퍼파라미터를 전달하여 NNLM 모델 객체를 생성합니다.
    model = NNLM(vocab_size, EMBEDDING_DIM, BLOCK_SIZE, HIDDEN_DIM)
    
    # 다중 클래스 분류 문제를 위해 Cross Entropy Loss를 손실 함수로 설정합니다.
    # 내부에 Softmax 연산과 Negative Log-Likelihood(NLL) 계산이 통합되어 있습니다.
    criterion = nn.CrossEntropyLoss()
    
    # 가중치 최적화를 위한 알고리즘으로 Adam 옵티마이저를 사용합니다.
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # NNLM 모델의 본격적인 학습 루프를 시작합니다.
    print("=== NNLM 학습 시작 ===")
    for epoch in range(1, EPOCHS + 1):
        # 이전 반복(Iteration)에서 계산된 경사도(Gradient)를 0으로 초기화합니다.
        optimizer.zero_grad()
        
        # 모델의 forward() 메서드를 호출하여 순전파를 진행하고 로짓을 얻습니다.
        logits = model(X)
        
        # 예측된 로짓과 실제 정답 타깃(Y) 사이의 손실(Loss)을 계산합니다.
        loss = criterion(logits, Y)
        
        # 역전파(Backpropagation)를 수행하여 각 파라미터(W, b)에 대한 기울기를 계산합니다.
        loss.backward()
        
        # 계산된 기울기 방향의 반대로 가중치 파라미터들을 업데이트합니다.
        optimizer.step()
        
        # 40 에폭마다 현재 학습 진행 상황 및 Loss 수치를 출력합니다.
        if epoch % 40 == 0:
            print(f"Epoch [{epoch:3d}/{EPOCHS}] - Loss: {loss.item():.4f}")

    # ==========================================
    # 4. 추론 (Inference / 예측) 테스트
    # ==========================================
    print("\n=== 추론 (다음 단어 예측) 테스트 ===")
    
    # 모델을 평가(Evaluation) 모드로 전환합니다. (드롭아웃, 배치 정규화 등의 동작 변경 대비)
    model.eval()
    
    # 추론 시에는 기울기(Gradient)를 계산할 필요가 없으므로 메모리 절약을 위해 autograd를 비활성화합니다.
    with torch.no_grad():
        # 테스트에 사용할 이전 3개 단어(Context) 시퀀스를 선언합니다.
        test_context_str = ["sequence", "of", "previous"]
        
        # 단어들을 대응되는 정수 인덱스로 변환하고, 모델 입력을 위해 배치 차원([1, 3])을 추가한 텐서를 만듭니다.
        test_context_idx = torch.tensor([[word2idx[w] for w in test_context_str]], dtype=torch.long)
        
        # 모델에 테스트 문맥을 입력하여 다음 단어 예측 로짓을 출력받습니다.
        output_logits = model(test_context_idx)
        
        # 가장 높은 로짓(확률)을 가진 단어의 인덱스를 선택(Argmax)합니다.
        predicted_idx = torch.argmax(output_logits, dim=-1).item()
        
        # 선택된 인덱스를 다시 실제 단어로 변환합니다.
        predicted_word = idx2word[predicted_idx]
        
        # 테스트 입력 문맥과 모델이 최종 예측한 다음 단어를 출력합니다.
        print(f"입력 문맥 (Context Window) : {' '.join(test_context_str)}")
        print(f"모델 예측 결과 (Next Word) : {predicted_word}")

words: ['the', 'neural', 'network', 'language', 'model', 'predicts', 'the', 'next', 'word', 'given', 'a', 'sequence', 'of', 'previous', 'words', 'using', 'embedding', 'vectors']
len(words): 18
=== NNLM 학습 시작 ===
embeds: tensor([[[-2.5095e+00,  4.8800e-01,  7.8459e-01,  2.8647e-02,  6.4076e-01,
           5.8325e-01,  1.0669e+00, -4.5015e-01],
         [-9.1382e-01, -6.5814e-01,  7.8024e-02,  5.2581e-01, -4.8799e-01,
           1.1914e+00, -8.1401e-01, -7.3599e-01],
         [-1.5576e+00,  9.9564e-01, -8.7979e-01, -6.0114e-01, -1.2742e+00,
           2.1228e+00, -1.2347e+00, -4.8791e-01]],

        [[-9.1382e-01, -6.5814e-01,  7.8024e-02,  5.2581e-01, -4.8799e-01,
           1.1914e+00, -8.1401e-01, -7.3599e-01],
         [-1.5576e+00,  9.9564e-01, -8.7979e-01, -6.0114e-01, -1.2742e+00,
           2.1228e+00, -1.2347e+00, -4.8791e-01],
         [ 1.2791e+00,  1.2964e+00,  6.1047e-01,  1.3347e+00, -2.3162e-01,
           4.1759e-02, -2.5158e-01,  8.5986e-01]],

        [[-1.5576e+00,  9.

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. NNLM (Neural Network Language Model) 클래스 정의
# ==========================================
class NNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size, hidden_dim):
        """
        NNLM 신경망의 레이어들을 초기화합니다.
        
        Args:
            vocab_size (int): 단어 집합(Vocabulary)의 전체 단어 개수
            embedding_dim (int): 원-핫 벡터를 투사할 밀집 임베딩 벡터의 차원 수
            block_size (int): 예측에 사용할 이전 단어의 개수 (Context Window Size = N - 1)
            hidden_dim (int): 은닉층(Hidden Layer)의 노드(뉴런) 개수
        """
        # 부모 클래스인 nn.Module의 초기화 메서드를 호출합니다.
        super(NNLM, self).__init__()
        
        # 1. 단어 인덱스를 embedding_dim 차원의 밀집 벡터로 변환하는 룩업 테이블(Look-up Table)입니다.
        # 단어장의 각 단어마다 고유한 임베딩 벡터가 매핑되며, 학습 과정에서 업데이트됩니다.
        self.C = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. 은닉층(Hidden Layer) 선형 결합 레이어입니다.
        # 이전 block_size개의 단어 임베딩이 하나로 이어붙여지므로(Concatenate),
        # 입력 차원은 (block_size * embedding_dim)이 됩니다.
        self.hidden = nn.Linear(block_size * embedding_dim, hidden_dim)
        
        # Bengio et al. (2003) 논문에서 사용한 비선형 활성화 함수 Tanh입니다.
        # 은닉층 연산 결과에 비선형성을 부여합니다.
        self.tanh = nn.Tanh()
        
        # 3. 은닉층의 출력을 받아 전체 단어장(vocab_size) 차원의 로짓(Logit)으로 변환하는 출력 레이어입니다.
        # 이 결과값에 Softmax를 취하면 다음 단어가 될 각 단어별 확률이 계산됩니다.
        self.output = nn.Linear(hidden_dim, vocab_size)
        
        # 4. Bengio 논문의 원본 구조에 존재하는 Direct Connection(직결 연결) 레이어입니다.
        # 은닉층을 거치지 않고, 임베딩 결합 벡터에서 바로 출력층으로 연결되는 가중치(W_x) 역할을 합니다.
        # 편향(bias)을 중복으로 더하지 않기 위해 bias=False로 설정합니다.
        self.direct = nn.Linear(block_size * embedding_dim, vocab_size, bias=False)

    def forward(self, x):
        """
        순전파(Forward Propagation) 연산을 수행합니다.
        
        Args:
            x (Tensor): [batch_size, block_size] 형태의 이전 단어 인덱스 텐서
            
        Returns:
            logits (Tensor): [batch_size, vocab_size] 형태의 다음 단어 예측 로짓
        """
        # 입력된 단어 인덱스들을 룩업하여 임베딩 벡터로 변환합니다.
        # 변환 후 텐서 크기: [batch_size, block_size, embedding_dim]
        embeds = self.C(x)
        
        # (N-1)개의 이전 단어 임베딩 벡터들을 차원 방향으로 평탄화하여 하나로 이어붙입니다(Concatenate).
        # 변환 후 텐서 크기: [batch_size, block_size * embedding_dim]
        concat_embeds = embeds.view(embeds.size(0), -1)
        
        # 선형 결합(self.hidden) 후 Tanh 활성화 함수를 통과시켜 은닉 상태(h)를 계산합니다.
        # 변환 후 텐서 크기: [batch_size, hidden_dim]
        h = self.tanh(self.hidden(concat_embeds))
        
        # Bengio 논문의 수식 y = b + Wx + U*tanh(d + Hx)를 계산합니다.
        # self.output(h)는 은닉층을 거친 경로(U*tanh...)를, self.direct(concat_embeds)는 직결 경로(Wx)를 의미합니다.
        # 변환 후 텐서 크기: [batch_size, vocab_size]
        logits = self.output(h) + self.direct(concat_embeds)
        
        # Softmax 직전의 값인 로짓(Logits)을 반환합니다.
        return logits


# ==========================================
# 2. 텍스트 데이터를 슬라이딩 윈도우로 전처리하는 함수
# ==========================================
def make_dataset(text, word2idx, block_size):
    """
    원문 텍스트를 입력 받아 NNLM 학습용 (입력 문맥 X, 정답 단어 Y) 텐서 쌍으로 변환합니다.
    """
    # 텍스트를 공백 기준으로 나누어 단어 리스트로 만듭니다.
    words = text.split()
    
    # 입력 문맥(X)과 타겟 단어(Y)를 담을 리스트 초기화
    X, Y = [], []
    
    # 전체 단어 시퀀스를 순회하며 슬라이딩 윈도우(Sliding Window)를 적용합니다.
    for i in range(len(words) - block_size):
        # 현재 위치 i부터 block_size(N-1)개까지의 이전 단어들을 정수 인덱스 리스트로 추출합니다.
        context = [word2idx[w] for w in words[i : i + block_size]]
        
        # 문맥 단어들 바로 다음에 오는 예측 대상 정답 단어의 인덱스를 추출합니다.
        target = word2idx[words[i + block_size]]
        
        # 리스트에 각각 추가합니다.
        X.append(context)
        Y.append(target)
        
    # PyTorch 모델에 입력할 수 있도록 정수형(Long) 텐서로 변환하여 반환합니다.
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)


# ==========================================
# 3. 모델 학습 및 검증 메인 파이프라인
# ==========================================
if __name__ == "__main__":
    # 무작위 추출 시 일관된 결과를 얻기 위해 PyTorch 난수 생성 시드를 고정합니다.
    torch.manual_seed(42)

    # NNLM 학습에 사용할 예시 코퍼스(Corpus) 문장입니다.
    sample_text = (
        "the neural network language model predicts the next word "
        "given a sequence of previous words using embedding vectors"
    )

    # 예시 문장을 단어 단위로 쪼개고 중복을 제거한 뒤 정렬하여 단어장 토큰 목록을 만듭니다.
    tokens = sorted(list(set(sample_text.split())))
    
    # 단어장의 전체 고유 단어 개수를 구합니다.
    vocab_size = len(tokens)
    
    # 단어를 정수 인덱스로 매핑하는 사전(Dictionary)을 생성합니다.
    word2idx = {w: i for i, w in enumerate(tokens)}
    
    # 정수 인덱스를 다시 단어로 복원할 때 사용하는 역방향 사전을 생성합니다.
    idx2word = {i: w for i, w in enumerate(tokens)}

    # 이전 몇 개의 단어(Context)를 보고 다음 단어를 예측할지 설정합니다. (N-1 = 3 -> N = 4)
    BLOCK_SIZE = 3
    
    # 각 단어를 몇 차원의 밀집 벡터로 임베딩할지 설정합니다.
    EMBEDDING_DIM = 8
    
    # 은닉층(Hidden Layer) 내부의 노드 개수를 설정합니다.
    HIDDEN_DIM = 16
    
    # 가중치 업데이트에 사용할 학습률(Learning Rate)을 설정합니다.
    LEARNING_RATE = 0.01
    
    # 전체 데이터셋을 총 몇 번 반복 학습할지 결정합니다.
    EPOCHS = 200

    # 텍스트 데이터로부터 NNLM 학습용 데이터셋(X: 문맥 인덱스, Y: 정답 인덱스)을 생성합니다.
    X, Y = make_dataset(sample_text, word2idx, BLOCK_SIZE)

    # 하이퍼파라미터를 전달하여 NNLM 모델 객체를 생성합니다.
    model = NNLM(vocab_size, EMBEDDING_DIM, BLOCK_SIZE, HIDDEN_DIM)
    
    # 다중 클래스 분류 문제를 위해 Cross Entropy Loss를 손실 함수로 설정합니다.
    # 내부에 Softmax 연산과 Negative Log-Likelihood(NLL) 계산이 통합되어 있습니다.
    criterion = nn.CrossEntropyLoss()
    
    # 가중치 최적화를 위한 알고리즘으로 Adam 옵티마이저를 사용합니다.
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # NNLM 모델의 본격적인 학습 루프를 시작합니다.
    print("=== NNLM 학습 시작 ===")
    for epoch in range(1, EPOCHS + 1):
        # 이전 반복(Iteration)에서 계산된 경사도(Gradient)를 0으로 초기화합니다.
        optimizer.zero_grad()
        
        # 모델의 forward() 메서드를 호출하여 순전파를 진행하고 로짓을 얻습니다.
        logits = model(X)
        
        # 예측된 로짓과 실제 정답 타깃(Y) 사이의 손실(Loss)을 계산합니다.
        loss = criterion(logits, Y)
        
        # 역전파(Backpropagation)를 수행하여 각 파라미터(W, b)에 대한 기울기를 계산합니다.
        loss.backward()
        
        # 계산된 기울기 방향의 반대로 가중치 파라미터들을 업데이트합니다.
        optimizer.step()
        
        # 40 에폭마다 현재 학습 진행 상황 및 Loss 수치를 출력합니다.
        if epoch % 40 == 0:
            print(f"Epoch [{epoch:3d}/{EPOCHS}] - Loss: {loss.item():.4f}")

    # ==========================================
    # 4. 추론 (Inference / 예측) 테스트
    # ==========================================
    print("\n=== 추론 (다음 단어 예측) 테스트 ===")
    
    # 모델을 평가(Evaluation) 모드로 전환합니다. (드롭아웃, 배치 정규화 등의 동작 변경 대비)
    model.eval()
    
    # 추론 시에는 기울기(Gradient)를 계산할 필요가 없으므로 메모리 절약을 위해 autograd를 비활성화합니다.
    with torch.no_grad():
        # 테스트에 사용할 이전 3개 단어(Context) 시퀀스를 선언합니다.
        test_context_str = ["sequence", "of", "previous"]
        
        # 단어들을 대응되는 정수 인덱스로 변환하고, 모델 입력을 위해 배치 차원([1, 3])을 추가한 텐서를 만듭니다.
        test_context_idx = torch.tensor([[word2idx[w] for w in test_context_str]], dtype=torch.long)
        
        # 모델에 테스트 문맥을 입력하여 다음 단어 예측 로짓을 출력받습니다.
        output_logits = model(test_context_idx)
        
        # 가장 높은 로짓(확률)을 가진 단어의 인덱스를 선택(Argmax)합니다.
        predicted_idx = torch.argmax(output_logits, dim=-1).item()
        
        # 선택된 인덱스를 다시 실제 단어로 변환합니다.
        predicted_word = idx2word[predicted_idx]
        
        # 테스트 입력 문맥과 모델이 최종 예측한 다음 단어를 출력합니다.
        print(f"입력 문맥 (Context Window) : {' '.join(test_context_str)}")
        print(f"모델 예측 결과 (Next Word) : {predicted_word}")

=== NNLM 학습 시작 ===
Epoch [ 40/200] - Loss: 0.0057
Epoch [ 80/200] - Loss: 0.0025
Epoch [120/200] - Loss: 0.0018
Epoch [160/200] - Loss: 0.0014
Epoch [200/200] - Loss: 0.0011

=== 추론 (다음 단어 예측) 테스트 ===
입력 문맥 (Context Window) : sequence of previous
모델 예측 결과 (Next Word) : words


In [8]:
import numpy as np

x = np.array([[1, 2, 3],
              [4, 5, 6]])  # Shape: [2, 3] (2차원)

In [9]:
sum_false = np.sum(x, axis=-1, keepdims=False)

print(sum_false)        # [6, 15]
print(sum_false.shape)  # (2,)  <- 2차원에서 1차원 벡터로 차원이 줄어듦!

[ 6 15]
(2,)


In [10]:
sum_false = np.sum(x, axis=-1, keepdims=True)

print(sum_false)        # [6, 15]
print(sum_false.shape)  # (2,)  <- 2차원에서 1차원 벡터로 차원이 줄어듦!

[[ 6]
 [15]]
(2, 1)


In [11]:
def softmax(x):
    # x shape: [2, 3]
    
    # 1. keepdims=True 사용 시
    # max_x shape: [2, 1]
    max_x = np.max(x, axis=-1, keepdims=True) 
    
    # [2, 3] - [2, 1]  ==>  브로드캐스팅 정상 작동!
    e_x = np.exp(x - max_x) 
    
    # sum_e_x shape: [2, 1]
    sum_e_x = np.sum(e_x, axis=-1, keepdims=True)
    
    # [2, 3] / [2, 1]  ==>  브로드캐스팅 정상 작동!
    return e_x / sum_e_x

In [12]:
import numpy as np

# 입력 벡터 x의 각 요소(Logit)에 대해 Softmax 확률 값을 계산하는 함수를 정의합니다.
# 행 단위(axis=-1)로 연산하여 각 배치 샘플별 확률의 합이 1이 되도록 만듭니다.
def softmax(x):
    # e_x shape: [batch_size, vocab_size] -> [2, 3]
    # np.max(x, axis=-1, keepdims=True)를 빼주어 지수 연산(exp) 시 발생하는 수치적 오버플로우(Overflow)를 방지합니다.
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    
    # return shape: [batch_size, vocab_size] -> [2, 3]
    # 지수화된 값들을 행 단위 합(sum)으로 나누어 모든 확률의 합이 1이 되도록 정규화한 값을 반환합니다.
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

# NumPy를 활용하여 바닐라 RNN 셀(Vanilla RNN Cell) 클래스를 정의합니다.
class VanillaRNNCellNumPy:
    # 클래스 생성자(초기화) 메서드: 입력, 은닉, 출력 차원 크기를 입력받아 가중치와 편향을 생성합니다.
    def __init__(self, input_dim: int, hidden_dim: int, vocab_size: int):
        """
        [전달된 파라미터 조건]
        - input_dim (input_size): 4
        - hidden_dim (hidden_size): 5
        - vocab_size (output_size): 3
        """
        
        # 1. 입력 x_t와 곱해지는 가중치 행렬 W_x 초기화
        # Shape: [input_dim, hidden_dim] -> [4, 5]
        # 무작위 난수 생성 후 0.01을 곱해 초기 연산값이 너무 커지는 것을 방지합니다.
        self.W_x = np.random.randn(input_dim, hidden_dim) * 0.01   
        
        # 2. 이전 은닉 상태 h_{t-1}과 곱해지는 가중치 행렬 W_h 초기화
        # Shape: [hidden_dim, hidden_dim] -> [5, 5]
        self.W_h = np.random.randn(hidden_dim, hidden_dim) * 0.01   
        
        # 3. 현재 은닉 상태 h_t와 곱해져 출력층으로 향하는 가중치 행렬 W_y 초기화
        # Shape: [hidden_dim, vocab_size] -> [5, 3]
        self.W_y = np.random.randn(hidden_dim, vocab_size) * 0.01   
        
        # 4. 은닉 상태 계산에 사용되는 편향(Bias) b_h를 0 벡터로 초기화
        # Shape: [1, hidden_dim] -> [1, 5]
        self.b_h = np.zeros((1, hidden_dim))                       
        
        # 5. 출력 계산에 사용되는 편향(Bias) b_y를 0 벡터로 초기화
        # Shape: [1, vocab_size] -> [1, 3]
        self.b_y = np.zeros((1, vocab_size))                       

    # 단일 시점(Time Step t)에서의 순전파(Forward Pass) 연산을 수행하는 메서드입니다.
    def forward(self, x_t: np.ndarray, h_prev: np.ndarray):
        """
        [입력 데이터 Shape] (batch_size = 2)
        :param x_t:    [batch_size, input_dim]  -> [2, 4]
        :param h_prev: [batch_size, hidden_dim] -> [2, 5]
        """        
        
        # -------------------------------------------------------------
        # [수식 1 계산]:  h_t = tanh(x_t . W_x + h_t-1 . W_h + b_h) 
        # -------------------------------------------------------------
        # np.dot(x_t, self.W_x):     현재 입력과 입력 가중치의 행렬 곱 [2, 4] x [4, 5] -> [2, 5]
        # np.dot(h_prev, self.W_h): 이전 은닉 상태와 은닉 가중치의 행렬 곱 [2, 5] x [5, 5] -> [2, 5]
        # self.b_h:                  [1, 5] 크기이며, 브로드캐스팅(Broadcasting)에 의해 [2, 5]로 자동 확장 반영
        # affine_h shape:            두 행렬 곱과 편향을 더한 선형 결합 결과 -> [2, 5]
        affine_h = np.dot(x_t, self.W_x) + np.dot(h_prev, self.W_h) + self.b_h
        
        # 선형 결합 결과(affine_h)에 비선형 활성화 함수인 hyperbolic tangent(tanh)를 적용합니다.
        # h_t shape: 출력이 -1과 1 사이로 압축된 현재 시점의 은닉 상태 -> [2, 5]
        h_t = np.tanh(affine_h)  
        
        # -------------------------------------------------------------
        # [수식 2 계산]:  y_t = Softmax(h_t . W_y + b_y) 
        # -------------------------------------------------------------
        # np.dot(h_t, self.W_y): 새로 계산된 은닉 상태 h_t와 출력 가중치 W_y의 행렬 곱 [2, 5] x [5, 3] -> [2, 3]
        # self.b_y:              [1, 3] 크기이며, 브로드캐스팅에 의해 [2, 3]으로 확장하여 합산
        # affine_y shape:        출력층 선형 결합 결과 (Logits) -> [2, 3]
        affine_y = np.dot(h_t, self.W_y) + self.b_y
        
        # 선형 결합 결과(affine_y)에 Softmax 함수를 적용하여 각 클래스별 예측 확률을 생성합니다.
        # y_t shape: 배치별 3개 클래스에 대한 확률 분포 (행 단위 합 = 1) -> [2, 3]
        y_t = softmax(affine_y)  
        
        # 현재 시점에서 계산된 은닉 상태(h_t)와 예측 확률 분포(y_t)를 반환합니다.
        return h_t, y_t


# 메인 스크립트 실행 및 동작 테스트 구간입니다.
if __name__ == "__main__":
    
    # 1개 배치에 포함되는 데이터 샘플 수 (Batch Size)
    batch_size = 2    
    # 입력 벡터 x_t의 특징(Feature) 개수 (Input Dimension)
    input_size = 4    
    # 모델 내부 은닉 상태 벡터의 차원 수 (Hidden Dimension)
    hidden_size = 5   
    # 최종 예측할 클래스/단어장의 개수 (Output Dimension)
    output_size = 3   
    
    # 인스턴스 파라미터 조건으로 NumPy RNN 셀 객체 생성
    rnn_cell = VanillaRNNCellNumPy(input_size, hidden_size, output_size)

    # 평균 0, 표준편차 1을 따르는 난수로 현재 시점 입력 x_t 생성 (Shape: [2, 4])
    x_t = np.random.randn(batch_size, input_size)      
    
    # 평균 0, 표준편차 1을 따르는 난수로 이전 시점 은닉 상태 h_prev 생성 (Shape: [2, 5])
    h_prev = np.random.randn(batch_size, hidden_size)  
    
    # forward 메서드를 호출하여 순전파 연산 수행 (h_t shape: [2, 5], y_t shape: [2, 3])
    h_t, y_t = rnn_cell.forward(x_t, h_prev)
    
    # 연산 결과 및 Shape 정보를 콘솔에 출력
    print("--- [출력 결과] ---")
    print(f"현재 입력 x_t 크기          : {x_t.shape}")       # (2, 4)
    print(f"이전 은닉 상태 h_{{t-1}} 크기 : {h_prev.shape}")   # (2, 5)
    print(f"현재 은닉 상태 h_t 크기     : {h_t.shape}")       # (2, 5)
    print(f"현재 출력 y_t 크기          : {y_t.shape}")       # (2, 3)
    
    # Softmax 확률 분포 결과 출력 (각 행의 합 = 1)
    print("\n최종 출력 y_t (Softmax 확률값):")
    print(y_t)

--- [출력 결과] ---
현재 입력 x_t 크기          : (2, 4)
이전 은닉 상태 h_{t-1} 크기 : (2, 5)
현재 은닉 상태 h_t 크기     : (2, 5)
현재 출력 y_t 크기          : (2, 3)

최종 출력 y_t (Softmax 확률값):
[[0.33321366 0.33341275 0.33337358]
 [0.33336984 0.33328957 0.33334059]]


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 바닐라 RNN 셀(Vanilla RNN Cell) 클래스 정의
# torch.nn.Module을 상속받아 PyTorch의 신경망 레이어로 동작합니다.
class VanillaRNNCell(nn.Module):
    """
    바닐라 RNN 셀 (Vanilla RNN Cell) 구현
    
    수식:
    1. Hidden State (은닉 상태) 계산:
       h_t = tanh(W_x * x_t + W_h * h_{t-1} + b_h)
    
    2. Output (출력값) 계산:
       y_t = softmax(W_y * h_t + b_y)
    """
    # 클래스 생성자(초기화) 메서드: 모델 생성 시 차원 정보들을 전달받아 가중치와 편향을 초기화합니다.
    def __init__(self, input_size: int, hidden_size: int, output_size: int):
        """
        [매개변수 설명]
        :param input_size: 입력 벡터 x_t의 차원 (Feature 수) -> 예: 4
        :param hidden_size: 은닉 상태 h_t의 차원 -> 예: 5
        :param output_size: 출력 벡터 y_t의 차원 (클래스 개수 / 단어 사전 크기 등) -> 예: 3
        """
        # 부모 클래스(nn.Module)의 생성자를 호출하여 PyTorch의 기본 레이어 기능들을 등록합니다.
        super(VanillaRNNCell, self).__init__()
        
        # 입력 차원 크기(input_size=4)를 인스턴스 변수에 저장합니다.
        self.input_size = input_size
        # 은닉 상태 차원 크기(hidden_size=5)를 인스턴스 변수에 저장합니다.
        self.hidden_size = hidden_size
        # 출력 차원 크기(output_size=3)를 인스턴스 변수에 저장합니다.
        self.output_size = output_size
        
        # -------------------------------------------------------------
        # 1. 은닉 상태(h_t) 계산을 위한 가중치 및 편향 선언
        # -------------------------------------------------------------
        
        # 입력 x_t와 곱해지는 가중치 W_x를 파라미터로 선언합니다.
        # PyTorch 표준 저장 방식 [out_features, in_features]에 따라 Shape은 [5, 4]가 됩니다.
        # 무작위 난수 생성 후 0.01을 곱해 초기 연산 값이 너무 커지는 것을 방지합니다.
        self.W_x = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)
        
        # 이전 은닉 상태 h_{t-1}과 곱해지는 가중치 W_h를 파라미터로 선언합니다.
        # Shape: [hidden_size, hidden_size] -> [5, 5]
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        
        # 은닉 상태 계산용 편향(Bias) b_h를 0으로 초기화하여 선언합니다.
        # Shape: [hidden_size, 1] -> [5, 1]
        self.b_h = nn.Parameter(torch.zeros(hidden_size, 1))
        
        # -------------------------------------------------------------
        # 2. 출력값(y_t) 계산을 위한 가중치 및 편향 선언
        # -------------------------------------------------------------
        
        # 은닉 상태 h_t와 곱해져 출력층으로 들어가는 가중치 W_y를 선언합니다.
        # Shape: [output_size, hidden_size] -> [3, 5]
        self.W_y = nn.Parameter(torch.randn(output_size, hidden_size) * 0.01)
        
        # 출력 계산용 편향(Bias) b_y를 0으로 초기화하여 선언합니다.
        # Shape: [output_size, 1] -> [3, 1]
        self.b_y = nn.Parameter(torch.zeros(output_size, 1))

    # 단일 시점(Time Step t)에서의 순전파(Forward Pass) 연산을 수행하는 메서드
    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor = None):
        """
        [순전파 연산]
        :param x_t: 현재 시점 t의 입력 (Batch Size=2, Input Size=4) -> Shape: [2, 4]
        :param h_prev: 이전 시점 t-1의 은닉 상태 (Batch Size=2, Hidden Size=5) -> Shape: [2, 5]
        :return: y_t (현재 시점 Softmax 확률), h_t (현재 시점 은닉 상태)
        """
        # 입력 데이터 x_t에서 배치 크기(Batch Size=2)를 추출합니다.
        batch_size = x_t.size(0)
        
        # 만약 이전 은닉 상태(h_prev)가 입력되지 않았다면 (첫 번째 시점 t=0인 경우)
        if h_prev is None:
            # 0으로 채워진 초기 은닉 상태 텐서를 생성합니다.
            # Shape: [batch_size, hidden_size] -> [2, 5]
            h_prev = torch.zeros(batch_size, self.hidden_size, device=x_t.device)
            
        # -------------------------------------------------------------
        # [식 1] 은닉 상태 계산: h_t = tanh(W_x * x_t + W_h * h_{t-1} + b_h)
        # -------------------------------------------------------------
        
        # 입력 데이터 x_t와 W_x의 전치 행렬(W_x.T)을 행렬 곱(matmul)합니다.
        # x_t: [2, 4], W_x.T: [4, 5] -> 연산 결과 wx_xt Shape: [2, 5]
        wx_xt = torch.matmul(x_t, self.W_x.T)
        
        # 이전 은닉 상태 h_prev와 W_h의 전치 행렬(W_h.T)을 행렬 곱(matmul)합니다.
        # h_prev: [2, 5], W_h.T: [5, 5] -> 연산 결과 wh_hprev Shape: [2, 5]
        wh_hprev = torch.matmul(h_prev, self.W_h.T)
        
        # 입력 선형결합(wx_xt) + 은닉 선형결합(wh_hprev) + 편향(b_h.T)을 더해 결합 연산을 완성합니다.
        # b_h.T: [1, 5]는 브로드캐스팅(Broadcasting)에 의해 [2, 5]로 자동 확장되어 더해집니다.
        # affine_h Shape: [2, 5] + [2, 5] + [1, 5] -> [2, 5]
        affine_h = wx_xt + wh_hprev + self.b_h.T
        
        # 선형 결합 결과에 비선형 활성화 함수인 hyperbolic tangent(tanh)를 적용합니다.
        # 출력을 -1과 1 사이로 압축하여 기울기 유지를 돕고, 현재 시점의 은닉 상태 h_t를 생성합니다.
        # h_t Shape: [2, 5]
        h_t = torch.tanh(affine_h)
        
        # -------------------------------------------------------------
        # [식 2] 출력값 계산: y_t = softmax(W_y * h_t + b_y)
        # -------------------------------------------------------------
        
        # 현재 은닉 상태 h_t에 출력 가중치 전치 행렬(W_y.T)을 곱하고 편향(b_y.T)을 더합니다.
        # h_t: [2, 5], W_y.T: [5, 3] -> [2, 3]
        # b_y.T: [1, 3] (브로드캐스팅 적용)
        # affine_y Shape: [2, 3]
        affine_y = torch.matmul(h_t, self.W_y.T) + self.b_y.T
        
        # 선형 결합 결과(Logits)의 마지막 차원(dim=-1)을 기준으로 Softmax를 적용합니다.
        # 각 클래스별 확률 값(0~1 사이, 행 단위 합=1) 분포 텐서 y_t를 생성합니다.
        # y_t Shape: [2, 3]
        y_t = F.softmax(affine_y, dim=-1)
        
        # 계산 완료된 현재 시점의 예측 확률 분포(y_t)와 은닉 상태(h_t)를 동시에 반환합니다.
        return y_t, h_t
    
# -------------------------------------------------------------
# 메인 스크립트 실행 구간 (코드 동작 테스트)
# -------------------------------------------------------------
if __name__ == "__main__":
    # 한 번에 처리할 데이터의 샘플 개수를 2로 설정합니다.
    batch_size = 2
    # 입력 데이터의 특징(Feature) 차원 수(input_size)를 4로 설정합니다.
    input_size = 4
    # 모델 내부 은닉 상태의 벡터 차원 수(hidden_size)를 5로 설정합니다.
    hidden_size = 5
    # 최종 출력 클래스(또는 단어장)의 개수(output_size)를 3으로 설정합니다.
    output_size = 3
    
    # 설정한 차원 값을 전달하여 VanillaRNNCell 인스턴스를 생성합니다.
    rnn_cell = VanillaRNNCell(input_size, hidden_size, output_size)

    # 표준정규분포를 따르는 가상의 현재 시점 입력 텐서 x_t를 생성합니다. (Shape: [2, 4])
    x_t = torch.randn(batch_size, input_size)
    
    # 표준정규분포를 따르는 가상의 이전 시점 은닉 상태 텐서 h_prev를 생성합니다. (Shape: [2, 5])
    h_prev = torch.randn(batch_size, hidden_size)
    
    # VanillaRNNCell의 forward 메서드를 호출하여 예측 확률(y_t)과 현재 은닉 상태(h_t)를 얻습니다.
    y_t, h_t = rnn_cell(x_t, h_prev)
    
    # 연산 결과 및 Shape 정보를 콘솔에 출력합니다.
    print("--- [출력 결과] ---")
    print(f"현재 입력 x_t 크기          : {x_t.shape}")        # torch.Size([2, 4])
    print(f"이전 은닉 상태 h_{{t-1}} 크기 : {h_prev.shape}")    # torch.Size([2, 5])
    print(f"현재 은닉 상태 h_t 크기     : {h_t.shape}")        # torch.Size([2, 5])
    print(f"현재 출력 y_t 크기          : {y_t.shape}")        # torch.Size([2, 3])
    
    # Softmax 확률 분포 값을 화면에 출력합니다.
    print("\n최종 출력 y_t (Softmax 확률값):")
    print(y_t)

--- [출력 결과] ---
현재 입력 x_t 크기          : torch.Size([2, 4])
이전 은닉 상태 h_{t-1} 크기 : torch.Size([2, 5])
현재 은닉 상태 h_t 크기     : torch.Size([2, 5])
현재 출력 y_t 크기          : torch.Size([2, 3])

최종 출력 y_t (Softmax 확률값):
tensor([[0.3331, 0.3336, 0.3333],
        [0.3333, 0.3333, 0.3334]], grad_fn=<SoftmaxBackward0>)


In [14]:
# 파이썬 표준 라이브러리 및 수학/수치 계산용 모듈 로드
import math
import re
import numpy as np
import pandas as pd
from collections import Counter

# PyTorch 프레임워크 관련 핵심 모듈 및 최적화 도구 로드
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 한글 형태소 분석을 위한 KoNLPy의 Okt 토크나이저 로드
from konlpy.tag import Okt

# 재현 가능성(Reproducibility) 확보를 위해 PyTorch 및 NumPy 난수 시드 고정
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 1. 데이터 로드 및 라벨링
# ==========================================

# Daum 영화 리뷰 CSV 파일을 읽어와 Pandas DataFrame 객체로 생성합니다.
df = pd.read_csv("data/daum_movie_review.csv")

# 감성 분류의 경계선이 모호한 6점과 7점 평점 리뷰 데이터를 제외하고 복사본을 생성합니다.
df_filtered = df[(df['rating'] != 6) & (df['rating'] != 7)].copy()

# 평점이 8점 이상인 경우 긍정(1), 5점 이하인 경우 부정(0)으로 분류하여 정수형 라벨 컬럼을 생성합니다.
df_filtered['label'] = (df_filtered['rating'] >= 8).astype(int)


# ==========================================
# 2. Okt 형태소 분석기 기반 전처리
# ==========================================

# KoNLPy의 Okt 형태소 분석기 인스턴스를 생성합니다.
okt = Okt()

# 입력 텍스트 전처리 및 형태소 분합을 수행하는 함수 정의
def okt_tokenize(text):
    # 정규표현식을 통해 한글, 영문, 공백을 제외한 모든 특수문자 및 숫자를 제거합니다.
    cleaned_text = re.sub(r'[^가-힣a-zA-Z\s]', '', str(text))
    
    # Okt 형태소 분석기를 사용해 문장을 형태소 단위로 토큰화합니다. (stem=True로 어간 추출)
    tokens = okt.morphs(cleaned_text, stem=True)
    
    # 공백 문자열을 제외한 유효한 형태소 토큰만 리스트로 반환합니다.
    tokens = [t for t in tokens if len(t.strip()) > 0]
    return tokens

print("=== Okt 형태소 분석기를 사용한 토큰화 진행 중... ===")
# 리뷰 데이터 컬럼 전체에 형태소 토큰화 함수를 적용하여 새로운 'tokens' 컬럼을 생성합니다.
df_filtered['tokens'] = df_filtered['review'].apply(okt_tokenize)

# 전처리 결과 형태소 토큰이 하나도 남지 않은 빈 리뷰 행을 필터링하여 제거합니다.
df_filtered = df_filtered[df_filtered['tokens'].apply(len) > 0].copy()

# 전체 리뷰에 등장한 모든 형태소 토큰을 하나의 리스트로 통합합니다.
all_tokens = [token for tokens in df_filtered['tokens'] for token in tokens]
# Counter를 활용해 각 형태소 토큰의 빈도수를 계산합니다.
token_counts = Counter(all_tokens)

# 패딩용 토큰(<PAD>: 0)과 미등록 단어용 토큰(<UNK>: 1)을 포함하는 단어장 사전(Vocabulary)을 정의합니다.
vocab = {'<PAD>': 0, '<UNK>': 1}

# 전체 데이터셋에서 2회 이상 등장한 형태소만 단어장에 고유 인덱스와 함께 등록합니다.
for token, count in token_counts.items():
    if count >= 2:
        vocab[token] = len(vocab)

# 형태소 토큰 리스트를 단어장 인덱스 번호의 리스트로 변환하고 최대 길이에 맞춰 패딩 처리하는 함수 정의
def tokens_to_ids(tokens, vocab, max_len=30):
    # 토큰이 단어장에 존재하면 해당 인덱스를, 없으면 <UNK>(1) 인덱스를 부여하고 최대 30개로 자릅니다.
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens[:max_len]]
    # 문장 길이가 max_len(30)보다 짧은 경우 <PAD>(0) 인덱스를 채워 길이를 맞춥니다.
    if len(ids) < max_len:
        ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

# 'tokens' 컬럼의 형태소 리스트를 길이 30의 정수 인덱스 리스트(input_ids)로 변환합니다.
# [Shape: (N, 30)]
df_filtered['input_ids'] = df_filtered['tokens'].apply(lambda x: tokens_to_ids(x, vocab, max_len=30))


# ==========================================
# 3. Train / Validation / Test 데이터 분할 (70 : 15 : 15)
# ==========================================

# 입력 데이터(X)와 라벨 데이터(y)를 NumPy 배열 형태로 변환합니다.
X = np.array(df_filtered['input_ids'].tolist()) # Shape: [N, 30]
y = np.array(df_filtered['label'].tolist())     # Shape: [N]

# 전체 데이터를 학습용(70%) 데이터와 임시 데이터(30%)로 계층적 분할(stratify)합니다.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 임시 30% 데이터를 검증용(15%) 데이터와 테스트용(15%) 데이터로 동일하게 분할합니다.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


# ==========================================
# 4. Dataset 및 DataLoader 생성
# ==========================================

# PyTorch Dataset 클래스를 상속받아 커스텀 데이터셋을 정의합니다.
class ReviewDataset(Dataset):
    # 생성자: NumPy 배열을 받아 PyTorch Tensor 형식으로 변환하여 저장합니다.
    def __init__(self, X, y):
        # X 텐서 Shape: [샘플 수, 30] (torch.long)
        self.X = torch.tensor(X, dtype=torch.long)
        # y 텐서 Shape: [샘플 수] (torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    # 데이터셋의 전체 샘플 개수를 반환합니다.
    def __len__(self):
        return len(self.X)
        
    # 지정한 인덱스(idx)에 해당하는 입력 데이터 샘플과 라벨을 반환합니다.
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Train, Validation, Test 데이터셋 인스턴스를 각각 생성합니다.
train_dataset = ReviewDataset(X_train, y_train)
val_dataset = ReviewDataset(X_val, y_val)
test_dataset = ReviewDataset(X_test, y_test)

# Mini-batch 처리를 위해 DataLoader 객체로 감싸줍니다. (배치 크기 = 64)
# train_loader 호출 시 배치는 batch_x: [64, 30], batch_y: [64] 형태로 추출됩니다.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


# ==========================================
# 5. Vanilla RNN 모델 정의 및 학습
# ==========================================

# PyTorch 기반 Vanilla RNN 감성 분류기 클래스 정의
class VanillaRNNClassifier(nn.Module):
    # 신경망 내 필요한 레이어 객체들을 선언합니다.
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(VanillaRNNClassifier, self).__init__()
        # 단어 인덱스를 임베딩 벡터로 변환하는 레이어선언
        # 파라미터 W_embed Shape: [vocab_size, embed_dim] -> [vocab_size, 64]
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # 순환 신경망(RNN) 레이어 선언
        # weight_ih_l0 Shape: [64, 64], weight_hh_l0 Shape: [64, 64]
        # bias_ih_l0 Shape: [64], bias_hh_l0 Shape: [64]
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='tanh')
        
        # 은닉 상태를 1차원 출력(Logit)으로 변환하는 선형 연결 레이어 선언
        # fc.weight Shape: [1, 64], fc.bias Shape: [1]
        self.fc = nn.Linear(hidden_dim, 1)
        
    # 순전파 연산 진행 함수
    def forward(self, x):
        # x Shape: [64, 30] (Batch Size=64, Max Len=30)
        
        # 1. 임베딩 레이어 통과
        # embedded Shape: [64, 30, 64] (Batch Size, Seq Len, Embed Dim)
        embedded = self.embedding(x)
        
        # 2. RNN 레이어 통과
        # output Shape: [64, 30, 64] (전체 타임스텝의 은닉 상태)
        # hidden Shape: [1, 64, 64] (num_layers=1, Batch Size=64, Hidden Dim=64)
        output, hidden = self.rnn(embedded)
        
        # 3. 마지막 타임스텝의 은닉 상태 추출 및 차원 축소
        # hidden.squeeze(0) Shape: [64, 64]
        
        # 4. 선형 연결 레이어로 감성 분류 Logit 값 계산
        # logits Shape: [64, 1]
        logits = self.fc(hidden.squeeze(0))
        
        # logits.squeeze(1) Shape: [64] (라벨 y의 크기 [64]와 일치시킴)
        return logits.squeeze(1)

# 학습을 수행할 연산 장치(GPU 또는 CPU)를 지정합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 모델 인스턴스 생성 후 해당 장치로 이관합니다.
model = VanillaRNNClassifier(vocab_size=len(vocab), embed_dim=64, hidden_dim=64).to(device)

# 이진 분류용 손실 함수 (Sigmoid + Binary Cross Entropy 결합형) 선언
criterion = nn.BCEWithLogitsLoss()
# 모델 매개변수 최적화를 위한 Adam 최적화기 선언 (학습률 0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 전체 에포크(Epoch) 수 설정
EPOCHS = 5

print("\n=== Okt 전처리 데이터 기반 바닐라 RNN 학습 시작 ===")

# 지정된 에포크 수만큼 반복 학습 및 검증 진행
for epoch in range(1, EPOCHS + 1):
    # 1. Train Phase (학습 모드 전환)
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    # 학습 데이터로더에서 미니배치 단위로 데이터를 꺼내어 순전파 및 역전파 수행
    for batch_x, batch_y in train_loader:
        # 텐서 데이터를 GPU/CPU 장치로 이관합니다.
        # batch_x Shape: [64, 30], batch_y Shape: [64]
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # 기울기(Gradient) 초기화
        optimizer.zero_grad()
        
        # 순전파 연산으로 Logit 반환 (logits Shape: [64])
        logits = model(batch_x)
        # Loss 손실 계산
        loss = criterion(logits, batch_y)
        
        # 역전파 연산을 통해 경사하강법 기울기 계산
        loss.backward()
        # 가중치 업데이트
        optimizer.step()
        
        # 배치별 Loss 누적
        train_loss += loss.item() * len(batch_y)
        # Sigmoid 활성화 후 0.5 이상이면 1(긍정), 미만이면 0(부정)으로 예측
        preds = (torch.sigmoid(logits) >= 0.5).float()
        # 정답 수 누적
        train_correct += (preds == batch_y).sum().item()
        train_total += len(batch_y)
        
    # Epoch 기준 학습 손실률 및 정확도 산출
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    
    # 2. Validation Phase (검증 모드 전환)
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    # 기울기 계산을 수행하지 않음 (메모리 절약 및 속도 향상)
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item() * len(batch_y)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
            
    # Epoch 기준 검증 손실률 및 정확도 산출
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    # 에포크별 결과 출력
    print(f"Epoch [{epoch:2d}/{EPOCHS}] | "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")


# ==========================================
# 6. Test 데이터셋 최종 평가
# ==========================================

# 모델 평가 모드 설정
model.eval()
test_loss, test_correct, test_total = 0, 0, 0

# 평가 진행
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        
        test_loss += loss.item() * len(batch_y)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        test_correct += (preds == batch_y).sum().item()
        test_total += len(batch_y)

# 최종 성능 평가지표 출력
print("\n=== Okt 형태소 분석 적용 모델 평가 결과 ===")
print(f"Test Loss    : {test_loss / test_total:.4f}")
print(f"Test Accuracy: {(test_correct / test_total) * 100:.2f}%")

=== Okt 형태소 분석기를 사용한 토큰화 진행 중... ===

=== Okt 전처리 데이터 기반 바닐라 RNN 학습 시작 ===
Epoch [ 1/5] | Train Loss: 0.5880 | Train Acc: 72.95% | Val Loss: 0.5840 | Val Acc: 73.34%
Epoch [ 2/5] | Train Loss: 0.5743 | Train Acc: 73.43% | Val Loss: 0.5752 | Val Acc: 73.18%
Epoch [ 3/5] | Train Loss: 0.5504 | Train Acc: 72.56% | Val Loss: 0.5729 | Val Acc: 72.83%
Epoch [ 4/5] | Train Loss: 0.5356 | Train Acc: 74.00% | Val Loss: 0.5611 | Val Acc: 72.78%
Epoch [ 5/5] | Train Loss: 0.5344 | Train Acc: 74.69% | Val Loss: 0.5575 | Val Acc: 72.52%

=== Okt 형태소 분석 적용 모델 평가 결과 ===
Test Loss    : 0.5564
Test Accuracy: 72.57%


In [15]:
# 파이썬 표준 라이브러리 및 수학/수치 계산용 모듈 로드
import math
import re
import numpy as np
import pandas as pd
from collections import Counter

# PyTorch 프레임워크 관련 핵심 모듈 및 최적화 도구 로드
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 한글 형태소 분석을 위한 KoNLPy의 Okt 토크나이저 로드
from konlpy.tag import Okt

# 재현 가능성(Reproducibility) 확보를 위해 PyTorch 및 NumPy 난수 시드 고정
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 1. 데이터 로드 및 라벨링
# ==========================================

# Daum 영화 리뷰 CSV 파일을 읽어와 Pandas DataFrame 객체로 생성합니다.
df = pd.read_csv("data/daum_movie_review.csv")

# 감성 분류의 경계선이 모호한 6점과 7점 평점 리뷰 데이터를 제외하고 복사본을 생성합니다.
df_filtered = df[(df['rating'] != 6) & (df['rating'] != 7)].copy()

# 평점이 8점 이상인 경우 긍정(1), 5점 이하인 경우 부정(0)으로 분류하여 정수형 라벨 컬럼을 생성합니다.
df_filtered['label'] = (df_filtered['rating'] >= 8).astype(int)


# ==========================================
# 2. Okt 형태소 분석기 기반 전처리
# ==========================================

# KoNLPy의 Okt 형태소 분석기 인스턴스를 생성합니다.
okt = Okt()

# 입력 텍스트 전처리 및 형태소 분합을 수행하는 함수 정의
def okt_tokenize(text):
    # 정규표현식을 통해 한글, 영문, 공백을 제외한 모든 특수문자 및 숫자를 제거합니다.
    cleaned_text = re.sub(r'[^가-힣a-zA-Z\s]', '', str(text))
    
    # Okt 형태소 분석기를 사용해 문장을 형태소 단위로 토큰화합니다. (stem=True로 어간 추출)
    tokens = okt.morphs(cleaned_text, stem=True)
    
    # 공백 문자열을 제외한 유효한 형태소 토큰만 리스트로 반환합니다.
    tokens = [t for t in tokens if len(t.strip()) > 0]
    return tokens

print("=== Okt 형태소 분석기를 사용한 토큰화 진행 중... ===")
# 리뷰 데이터 컬럼 전체에 형태소 토큰화 함수를 적용하여 새로운 'tokens' 컬럼을 생성합니다.
df_filtered['tokens'] = df_filtered['review'].apply(okt_tokenize)

# 전처리 결과 형태소 토큰이 하나도 남지 않은 빈 리뷰 행을 필터링하여 제거합니다.
df_filtered = df_filtered[df_filtered['tokens'].apply(len) > 0].copy()

# 전체 리뷰에 등장한 모든 형태소 토큰을 하나의 리스트로 통합합니다.
all_tokens = [token for tokens in df_filtered['tokens'] for token in tokens]
# Counter를 활용해 각 형태소 토큰의 빈도수를 계산합니다.
token_counts = Counter(all_tokens)

# 패딩용 토큰(<PAD>: 0)과 미등록 단어용 토큰(<UNK>: 1)을 포함하는 단어장 사전(Vocabulary)을 정의합니다.
vocab = {'<PAD>': 0, '<UNK>': 1}

# 전체 데이터셋에서 2회 이상 등장한 형태소만 단어장에 고유 인덱스와 함께 등록합니다.
for token, count in token_counts.items():
    if count >= 2:
        vocab[token] = len(vocab)

# 형태소 토큰 리스트를 단어장 인덱스 번호의 리스트로 변환하고 최대 길이에 맞춰 패딩 처리하는 함수 정의
def tokens_to_ids(tokens, vocab, max_len=30):
    # 토큰이 단어장에 존재하면 해당 인덱스를, 없으면 <UNK>(1) 인덱스를 부여하고 최대 30개로 자릅니다.
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens[:max_len]]
    # 문장 길이가 max_len(30)보다 짧은 경우 <PAD>(0) 인덱스를 채워 길이를 맞춥니다.
    if len(ids) < max_len:
        ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

# 'tokens' 컬럼의 형태소 리스트를 길이 30의 정수 인덱스 리스트(input_ids)로 변환합니다.
# [Shape: (N, 30)]
df_filtered['input_ids'] = df_filtered['tokens'].apply(lambda x: tokens_to_ids(x, vocab, max_len=30))


# ==========================================
# 3. Train / Validation / Test 데이터 분할 (70 : 15 : 15)
# ==========================================

# 입력 데이터(X)와 라벨 데이터(y)를 NumPy 배열 형태로 변환합니다.
X = np.array(df_filtered['input_ids'].tolist()) # Shape: [N, 30]
y = np.array(df_filtered['label'].tolist())     # Shape: [N]

# 전체 데이터를 학습용(70%) 데이터와 임시 데이터(30%)로 계층적 분할(stratify)합니다.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 임시 30% 데이터를 검증용(15%) 데이터와 테스트용(15%) 데이터로 동일하게 분할합니다.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


# ==========================================
# 4. Dataset 및 DataLoader 생성
# ==========================================

# PyTorch Dataset 클래스를 상속받아 커스텀 데이터셋을 정의합니다.
class ReviewDataset(Dataset):
    # 생성자: NumPy 배열을 받아 PyTorch Tensor 형식으로 변환하여 저장합니다.
    def __init__(self, X, y):
        # X 텐서 Shape: [샘플 수, 30] (torch.long)
        self.X = torch.tensor(X, dtype=torch.long)
        # y 텐서 Shape: [샘플 수] (torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    # 데이터셋의 전체 샘플 개수를 반환합니다.
    def __len__(self):
        return len(self.X)
        
    # 지정한 인덱스(idx)에 해당하는 입력 데이터 샘플과 라벨을 반환합니다.
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Train, Validation, Test 데이터셋 인스턴스를 각각 생성합니다.
train_dataset = ReviewDataset(X_train, y_train)
val_dataset = ReviewDataset(X_val, y_val)
test_dataset = ReviewDataset(X_test, y_test)

# Mini-batch 처리를 위해 DataLoader 객체로 감싸줍니다. (배치 크기 = 64)
# train_loader 호출 시 배치는 batch_x: [64, 30], batch_y: [64] 형태로 추출됩니다.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


# ==========================================
# 5. Vanilla RNN 모델 정의 및 학습
# ==========================================

# PyTorch 기반 Vanilla RNN 감성 분류기 클래스 정의
class VanillaRNNClassifier(nn.Module):
    # 신경망 내 필요한 레이어 객체들을 선언합니다.
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(VanillaRNNClassifier, self).__init__()
        # 단어 인덱스를 임베딩 벡터로 변환하는 레이어선언
        # 파라미터 W_embed Shape: [vocab_size, embed_dim] -> [vocab_size, 64]
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # 순환 신경망(RNN) 레이어 선언
        # weight_ih_l0 Shape: [64, 64], weight_hh_l0 Shape: [64, 64]
        # bias_ih_l0 Shape: [64], bias_hh_l0 Shape: [64]
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='tanh')
        
        # 은닉 상태를 1차원 출력(Logit)으로 변환하는 선형 연결 레이어 선언
        # fc.weight Shape: [1, 64], fc.bias Shape: [1]
        self.fc = nn.Linear(hidden_dim, 1)
        
    # 순전파 연산 진행 함수
    def forward(self, x):
        # x Shape: [64, 30] (Batch Size=64, Max Len=30)
        
        # 1. 임베딩 레이어 통과
        # embedded Shape: [64, 30, 64] (Batch Size, Seq Len, Embed Dim)
        embedded = self.embedding(x)
        
        # 2. RNN 레이어 통과
        # output Shape: [64, 30, 64] (전체 타임스텝의 은닉 상태)
        # hidden Shape: [1, 64, 64] (num_layers=1, Batch Size=64, Hidden Dim=64)
        output, hidden = self.rnn(embedded)
        
        # 3. 마지막 타임스텝의 은닉 상태 추출 및 차원 축소
        # hidden.squeeze(0) Shape: [64, 64]
        
        # 4. 선형 연결 레이어로 감성 분류 Logit 값 계산
        # logits Shape: [64, 1]
        logits = self.fc(hidden.squeeze(0))
        
        # logits.squeeze(1) Shape: [64] (라벨 y의 크기 [64]와 일치시킴)
        return logits.squeeze(1)

# 학습을 수행할 연산 장치(GPU 또는 CPU)를 지정합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 모델 인스턴스 생성 후 해당 장치로 이관합니다.
model = VanillaRNNClassifier(vocab_size=len(vocab), embed_dim=64, hidden_dim=64).to(device)

# 이진 분류용 손실 함수 (Sigmoid + Binary Cross Entropy 결합형) 선언
criterion = nn.BCEWithLogitsLoss()
# 모델 매개변수 최적화를 위한 Adam 최적화기 선언 (학습률 0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 전체 에포크(Epoch) 수 설정
EPOCHS = 5

print("\n=== Okt 전처리 데이터 기반 바닐라 RNN 학습 시작 ===")

# 지정된 에포크 수만큼 반복 학습 및 검증 진행
for epoch in range(1, EPOCHS + 1):
    # 1. Train Phase (학습 모드 전환)
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    # 학습 데이터로더에서 미니배치 단위로 데이터를 꺼내어 순전파 및 역전파 수행
    for batch_x, batch_y in train_loader:
        # 텐서 데이터를 GPU/CPU 장치로 이관합니다.
        # batch_x Shape: [64, 30], batch_y Shape: [64]
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # 기울기(Gradient) 초기화
        optimizer.zero_grad()
        
        # 순전파 연산으로 Logit 반환 (logits Shape: [64])
        logits = model(batch_x)
        # Loss 손실 계산
        loss = criterion(logits, batch_y)
        
        # 역전파 연산을 통해 경사하강법 기울기 계산
        loss.backward()
        # 가중치 업데이트
        optimizer.step()
        
        # 배치별 Loss 누적
        train_loss += loss.item() * len(batch_y)
        # Sigmoid 활성화 후 0.5 이상이면 1(긍정), 미만이면 0(부정)으로 예측
        preds = (torch.sigmoid(logits) >= 0.5).float()
        # 정답 수 누적
        train_correct += (preds == batch_y).sum().item()
        train_total += len(batch_y)
        
    # Epoch 기준 학습 손실률 및 정확도 산출
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    
    # 2. Validation Phase (검증 모드 전환)
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    # 기울기 계산을 수행하지 않음 (메모리 절약 및 속도 향상)
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item() * len(batch_y)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
            
    # Epoch 기준 검증 손실률 및 정확도 산출
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    # 에포크별 결과 출력
    print(f"Epoch [{epoch:2d}/{EPOCHS}] | "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")


# ==========================================
# 6. Test 데이터셋 최종 평가
# ==========================================

# 모델 평가 모드 설정
model.eval()
test_loss, test_correct, test_total = 0, 0, 0

# 평가 진행
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        
        test_loss += loss.item() * len(batch_y)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        test_correct += (preds == batch_y).sum().item()
        test_total += len(batch_y)

# 최종 성능 평가지표 출력
print("\n=== Okt 형태소 분석 적용 모델 평가 결과 ===")
print(f"Test Loss    : {test_loss / test_total:.4f}")
print(f"Test Accuracy: {(test_correct / test_total) * 100:.2f}%")

=== Okt 형태소 분석기를 사용한 토큰화 진행 중... ===

=== Okt 전처리 데이터 기반 바닐라 RNN 학습 시작 ===
Epoch [ 1/5] | Train Loss: 0.5880 | Train Acc: 72.95% | Val Loss: 0.5840 | Val Acc: 73.34%
Epoch [ 2/5] | Train Loss: 0.5743 | Train Acc: 73.43% | Val Loss: 0.5752 | Val Acc: 73.18%
Epoch [ 3/5] | Train Loss: 0.5504 | Train Acc: 72.56% | Val Loss: 0.5729 | Val Acc: 72.83%
Epoch [ 4/5] | Train Loss: 0.5356 | Train Acc: 74.00% | Val Loss: 0.5611 | Val Acc: 72.78%
Epoch [ 5/5] | Train Loss: 0.5344 | Train Acc: 74.69% | Val Loss: 0.5575 | Val Acc: 72.52%

=== Okt 형태소 분석 적용 모델 평가 결과 ===
Test Loss    : 0.5564
Test Accuracy: 72.57%
